# Imports

In [1]:
import json
import os
import time

import pandas as pd

from pathlib import Path
from dotenv import load_dotenv
from groq import Groq

# Paths

In [2]:
PROJECT_ROOT = Path.cwd().parent

PAPERS_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "arxiv_search.csv"
)

QUERY_CONFIG_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "arxiv_query.json"
)

OUTPUT_DIR = PROJECT_ROOT / "data" / "intermediate"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = OUTPUT_DIR / "paper_analyses.csv"

MODEL = "llama-3.1-8b-instant"
MAX_PAPERS = 5

In [3]:
load_dotenv(PROJECT_ROOT / ".env")

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

client = Groq(
    api_key=GROQ_API_KEY
)

In [4]:
df_papers = pd.read_csv(PAPERS_PATH)

In [5]:
with open(
    QUERY_CONFIG_PATH,
    encoding="utf-8",
) as file:
    query_config = json.load(file)

research_topic = query_config["topic"]
arxiv_query = query_config["query"]

print("Research topic:")
print(research_topic)

print("\narXiv query:")
print(arxiv_query)

Research topic:
i want to investigate about the gaussian distribution

arXiv query:
(all:"gaussian distribution" OR all:"gaussian function" OR all:"normal distribution") AND all:"probability theory"


In [6]:
df_selected = df_papers.head(MAX_PAPERS).copy()

# Prompting

In [7]:
analyses = []

for _, paper in df_selected.iterrows():

    prompt = f"""
You are the Analyst Agent in an academic research system.

The user's research topic is:

{research_topic}

Analyze the following paper using only the title and abstract provided.

Paper title:
{paper["title"]}

Abstract:
{paper["summary"]}

Return only a valid JSON object with these fields:

{{
  "summary": "A concise explanation of the paper",
  "main_problem": "The main problem addressed by the paper",
  "main_contribution": "The main contribution of the paper",
  "applications": "Potential practical applications",
  "limitations": "Limitations that can reasonably be inferred from the abstract",
  "relevance_score": 8,
  "relevance_reason": "Why the paper is or is not relevant to the user's topic"
}}

Rules:

- relevance_score must be an integer between 1 and 10.
- Do not include information that is not supported by the abstract.
- Do not use Markdown.
- Return only JSON.
"""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
        response_format={
            "type": "json_object"
        },
        temperature=0.2,
    )

    raw_analysis = response.choices[0].message.content

    analysis = json.loads(raw_analysis)

    analysis["title"] = paper["title"]
    analysis["authors"] = paper["authors"]
    analysis["published"] = paper["published"]
    analysis["url"] = paper["url"]
    analysis["original_abstract"] = paper["summary"]

    analyses.append(analysis)

    print(
        f'Analyzed: {paper["title"][:70]}...'
    )

    time.sleep(1)

Analyzed: A Constructive Approach to $q$-Gaussian Distributions: $α$-Divergence ...


Analyzed: Revisiting De Moivre-Laplace...


Analyzed: Topics in Probability, Parametric Estimation and Stochastic Calculus...


Analyzed: The fast rate of convergence of the smooth adapted Wasserstein distanc...


Analyzed: Probabilistic interpretation of the Selberg--Delange Method in analyti...


In [8]:
df_analyses = pd.DataFrame(analyses)

In [9]:
df_analyses = (
    df_analyses
    .sort_values(
        "relevance_score",
        ascending=False,
    )
    .reset_index(drop=True)
)

In [10]:
df_analyses.to_csv(
    OUTPUT_PATH,
    index=False,
)

print(f"Analyses saved to:\n{OUTPUT_PATH}")

Analyses saved to:
C:\Users\jortialo\ai-paper-review-agentic\data\intermediate\paper_analyses.csv


In [11]:
best_paper = df_analyses.iloc[0]

print("TITLE")
print(best_paper["title"])

print("\nRELEVANCE SCORE")
print(best_paper["relevance_score"])

print("\nSUMMARY")
print(best_paper["summary"])

print("\nMAIN CONTRIBUTION")
print(best_paper["main_contribution"])

print("\nRELEVANCE")
print(best_paper["relevance_reason"])

TITLE
A Constructive Approach to $q$-Gaussian Distributions: $α$-Divergence as Rate Function and Generalized de Moivre-Laplace Theorem

RELEVANCE SCORE
9

SUMMARY
The paper presents a constructive approach to derive power-law distributions, specifically the q-Gaussian distribution, using a nonlinear differential equation and finite counting.

MAIN CONTRIBUTION
The main contribution of the paper is a generalized binomial distribution based on finite counting, which leads to a q-Gaussian distribution, and the identification of the α-divergence as the rate function.

RELEVANCE
The paper is highly relevant to the user's topic, as it investigates the q-Gaussian distribution, which is a specific type of power-law distribution.


# Get response from local model